# Adição das informações macroscópicas geradas pelas interações microscópicas das partículas

Neste notebook, será introduzido o conceito de temperatura dentro da simulação. Segundo a Teoria Cinética dos Gases, a temperatura está diretamente relacionada à energia cinética média das partículas. Dessa forma, ao analisar as velocidades das partículas presentes no sistema, é possível obter uma medida da temperatura do gás.

Além do cálculo dessa grandeza, serão implementados mecanismos de controle que permitam alterar a energia do sistema e observar, em tempo real, os efeitos dessas mudanças sobre o comportamento coletivo das partículas. Essa abordagem transforma a simulação em um pequeno laboratório virtual, permitindo relacionar conceitos teóricos da Física com observações computacionais.

Ademais, as partículas serão diferenciadas por cor de acordo com a sua velocidade.

Essa simulação também mostra que propriedades macroscópicas podem emergir do comportamento coletivo de muitas partículas microscópicas.

In [51]:
import pygame as pyg
import random as rd
import math as mt

In [61]:
def cria_particulas(particulas, num_particulas):
    """Função que cria as particulas, recebendo uma lista vazia e um número de particulas para a simulação. Ela retorna a lista com n particulas e as suas propriedades (velocidade e posição)."""
    #Criação do número dado de partículas
    for _ in range(num_particulas):
        #Angulo de direção alearório para cada partícula
        angulo = rd.uniform(0, 2 * mt.pi)
        
        velocidade = rd.uniform(2, 6) #Velocidade aleatória para cada partícula
        
        #Direção x aleatoria
        x = rd.randint(
        caixa_x + raio,
        caixa_x + caixa_largura - raio
        )
        #Direção y aleatória
        y = rd.randint(
        caixa_y + raio,
        caixa_y + caixa_altura - raio
        )
        #Velocidades iguals, porém em direções aleatórias
        vx = velocidade*mt.cos(angulo)
        vy = velocidade*mt.sin(angulo)
        
        #Conjunto dentro de uma lista
        particulas.append(
        {
            "x": x,
            "y": y,
            "vx": vx,
            "vy": vy
        }
        )
    
    return particulas

In [62]:
def colisoes_parede(particulas):
    """Função que recebe a lista das particulas e inverte o sentido das suas velocidades quando elas colidem com a parede (colisões elasticas, retornando as velocidades invertidas das particulas."""
    for particula in particulas:
        #Criação das colisões para o teto e para o chão (lados direito e esquerdo)
    
        #Parede esquerda
        if particula["x"] <= caixa_x + raio:

            particula["x"] = caixa_x + raio

            particula["vx"] *= -1
        #parede direita
        if particula["x"] >= caixa_x + caixa_largura - raio:
            particula["x"] = caixa_x + caixa_largura - raio
            particula["vx"] *= -1
    
        #Teto superior
        if particula["y"] <= caixa_y + raio:
            particula["y"] = caixa_y + raio
            particula["vy"] *= -1
    
        #Teto inferior
        if particula["y"] >= caixa_y + caixa_altura - raio:
            particula["y"] = caixa_y + caixa_altura - raio
            particula["vy"] *= -1

In [63]:
def colisoes_particulas(particulas, raio):
    """Função que recebe a lista com as informações das partículas e detecta as colisões entre elas. Nesse código, as suas velocidades são invertidas (colisões elasticas) e um vetor de sobreposição normalizado separa elas imperceptívelmente para que uma nova colisão seja detectada (ler o markdown ao final do código)."""
    
    #Percorrer as partículas da lista e comparar as suas posições para as colisões
    for i in range(len(particulas)):
        #i+1, pois irá partir da próxima posição para vasculhar
        for j in range(i+1, len(particulas)):
            p1 = particulas[i]
            p2 = particulas[j]

            #distancia entre as partículas
            dx = p2["x"] - p1["x"]
            dy = p2["y"] - p1["y"]
            distancia = mt.sqrt((dx**2) + (dy**2))

            #caso elas se encostem (colisão), as velocidades são trocadas (colisão elastica)
            if distancia <= 2*raio and distancia > 0:
                p1["vx"], p2["vx"] = p2["vx"], p1["vx"]
                p1["vy"], p2["vy"] = p2["vy"], p1["vy"]

                #Separar minimamente as particulas para que colisões não sejam infinitas
                
                #normaliza os vetores de distancia entre as particulas
                nx = dx/distancia
                ny = dy/distancia

                #Corrige a sobreposição das partículas, ou seja, move cada partícula minimamente
                sobreposicao = 2*raio - distancia

                #Atualiza a posição das partículas, movendo-as minimamente para longe de si para que outra colisão não seja detectada
                p1["x"] -= nx * sobreposicao / 2
                p1["y"] -= ny * sobreposicao / 2
                p2["x"] += nx * sobreposicao / 2
                p2["y"] += ny * sobreposicao / 2

In [64]:
def mover_particulas(particulas):
    """Função que soma a velocidade de cada partícula à sua posição, dando a ideia de movimento."""
    #Atualiza a posição de cada particula (ideia de movimento)
    for particula in particulas:
        particula["x"] += particula["vx"]
        particula["y"] += particula["vy"]

In [65]:
def desenha_particulas(particulas, raio, tela):
    """Função que percorre as particulas e desenha as particulas na tela, recebendo a lista das particulas, o valor do seu raio e a tela da simulação."""

    #desenha as partículas na tela
    for particula in particulas:
        
        #Define a rapidez da partícula
        v = (particula["vx"]**2 + particula["vy"]**2)**0.5
        
        #Define a cor da partícula de acordo com a sua velocidade
        if v < 2:
            cor = (0, 0, 255)

        elif v < 4:
            cor = (0, 255, 0)

        else:
            cor = (255, 0, 0)
            
        #desenha a bolinha
        pyg.draw.circle(
            tela,
            cor,
            (
                int(particula["x"]),
                int(particula["y"])
            ),
            raio
        ) 

In [66]:
def desenha_caixa(tela, caixa_x, caixa_y, caixa_largura, caixa_altura):
    """Função que recebe as váriaveis das dimensões da caixa e retorna um desenho na tela da caixa com as dimensões proporcionadas."""

    #Desenho do recipiente
    pyg.draw.rect(
        tela,
        (20, 20, 20),
        (
            caixa_x,
            caixa_y,
            caixa_largura,
            caixa_altura
        ),
        width=2
    )
    

In [67]:
def texto_particulas(tela, fonte, particulas):
    """Função que recebe a fonte escolhida e a lista de partículas, retornando a escrita do número de particulas na tela."""
    
    #Mostra o número de partículas na tela
    texto = fonte.render(
    f"N = {len(particulas)}",
    True,
    (0, 0, 0)
    )
    tela.blit(texto, (620, 50))

In [68]:
def trata_evento(particulas, rodando):
    """ Gerencia os eventos do Pygame durante a simulação, permitindo encerrar o programa e ajustar a velocidade das partículas. A tecla UP aumenta a velocidade em 10% e a tecla DOWN reduz em 10%. Retorna o estado atualizado da execução. """
    
    #Captura eventos
    for evento in pyg.event.get():
        
        #Se o evento for sair do aplicativo, encerra-se o while e,consequentemente, a simulação
        if evento.type == pyg.QUIT:
            rodando = False

        #Caso a tecla para baixo seja pressionada, diminui-se a velocidade em 10%
        if evento.type == pyg.KEYDOWN:
            if evento.key == pyg.K_DOWN:
                for particula in particulas:

                    particula["vx"] *= 0.9
                    particula["vy"] *= 0.9    
        
        #Caso a tecla para cima seja pressionada, aumenta-se em 10% a velocidade
            if evento.key == pyg.K_UP:
                for particula in particulas:

                    particula["vx"] *= 1.1
                    particula["vy"] *= 1.1
    return rodando

In [77]:
def temp(particulas, fonte, tela):
    """ Calcula a temperatura média do sistema a partir das velocidades das partículas e exibe o valor correspondente na tela. Recebe a lista de partículas, a fonte utilizada para renderização do texto e a superfície onde a temperatura será desenhada. Não retorna nenhum valor, apenas executa os comandos de criação da temperatura e do texto que ilustra seu valor na tela."""
    
    #Criação da variavel temperatura
    temperatura = 0
    #Soma das velocidades ao quadrado de todas as partículas
    for particula in particulas:
        vx = particula["vx"]
        vy = particula["vy"]
        temperatura += vy**2 + vx**2

    #Calculo da energia cinetica média das particulas, isto é, a temperatura
    temperatura = temperatura/len(particulas)

    #Arredondando o seu valor para duas casas decimais
    temperatura = round(temperatura, 2)
    
    #Mostrando a temperatura na tela
    texto_temperatura = fonte.render(
    f"T = {temperatura}",
    True,
    (0,0,0)
    )

    #Mostrando na tela um pouco abaixo de N
    tela.blit(texto_temperatura, (620, 90))

In [78]:
def controles(fonte, tela):
      """ Exibe na tela os controles disponíveis para aquecer e resfriar o sistema. Recebe a fonte utilizada para renderizar os textos e a parte da tela onde eles serão desenhados. """
    
    texto_controle1 = fonte.render(
    "UP - Aquecer",
    True,
    (0,0,0)
    )

    texto_controle2 = fonte.render(
    "DOWN - Resfriar",
    True,
    (0,0,0)
    )
    
    #posicioná-los logo abaicxo de N e da temperatura.
    tela.blit(texto_controle1, (620, 140))
    tela.blit(texto_controle2, (620, 180))

In [85]:
pyg.init() #inicializa o pygame
tela = pyg.display.set_mode((800,600)) #Cria a janela e define o seu tamanho

#Título da janela
pyg.display.set_caption("Ideal Gas Lab")

#loop principal
rodando = True

#Definindo a variável clock como
clock = pyg.time.Clock()


#Criação do recipiente do gás
caixa_x = 25
caixa_y = 25

caixa_largura = 550
caixa_altura = 550

#Definindo a quantidade e o raio das partículas
raio = 10
num_particulas = 100

#Definindo a fonte que mostrará a mensagem na tela.
fonte = pyg.font.Font(None, 30)

#Criação das partículas
particulas = []

#Criação do número dado de partículas
cria_particulas(particulas, num_particulas)


#Loop em que a simulação ocorrerá
while rodando:
    

    #Captura eventos
    rodando = trata_evento(particulas, rodando)
            
    #Deixa o fundo cinza
    tela.fill((230, 230, 230))

    #Mostra o número de partículas na tela
    texto_particulas(tela, fonte, particulas)

    #Atualiza a posição de cada particula (ideia de movimento)
    mover_particulas(particulas)
    
    #Desenho do recipiente
    desenha_caixa(tela, caixa_x, caixa_y, caixa_largura, caixa_altura)
    
    #desenha as partículas na tela
    desenha_particulas(particulas, raio, tela)
        
    #colisões das particulas para o teto e para o chão (lados direito e esquerdo)
    colisoes_parede(particulas)

    #colisões das particulas umas com as outras
    colisoes_particulas(particulas, raio)

    #Chama a função temperatura que cria um valor arbitrário de temperatura de acordo com a velocidade média das partículas
    temp(particulas, fonte, tela)
    
    #Criar controles para aumentar e abaixar a temperatura
    controles(fonte, tela)

    
    #60 iterações por segundo (60 fps)
    clock.tick(60)
    
    #Atualiza a tela
    pyg.display.flip()




pyg.quit()